In [4]:
import numpy as np
import time
from numba import cuda

def double_cpu_pure(arr):
    out = np.empty_like(arr)
    for i in range(len(arr)):
        out[i] = arr[i] * 2.0
    return out

@cuda.jit
def double_gpu_kernel(d_in, d_out):
    idx = cuda.grid(1)
    if idx < d_in.shape[0]:
        d_out[idx] = d_in[idx] * 2.0

def benchmark_run(N):
    h_in = np.random.rand(N).astype(np.float32)

    if N <= 1_000_000:
        t0 = time.perf_counter()
        double_cpu_pure(h_in)
        cpu_ms = (time.perf_counter() - t0) * 1000
    else:
        cpu_ms = float("nan")

    threads = 256
    blocks = (N + threads - 1) // threads

    d_warm = cuda.to_device(np.ones(10, dtype=np.float32))
    double_gpu_kernel[1, 32](d_warm, d_warm)
    cuda.synchronize()

    t0 = time.perf_counter()
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array_like(d_in)
    double_gpu_kernel[blocks, threads](d_in, d_out)
    h_out = d_out.copy_to_host()
    cuda.synchronize()
    total_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    double_gpu_kernel[blocks, threads](d_in, d_out)
    cuda.synchronize()
    kernel_ms = (time.perf_counter() - t0) * 1000

    return cpu_ms, total_ms, kernel_ms

sizes = [100, 1000, 10000, 100000, 1000000, 5000000, 10000000]

print("N | CPU ms | GPU Total ms | GPU Kernel ms | Winner")
for N in sizes:
    cpu, total, kernel = benchmark_run(N)
    winner = "GPU" if total < cpu else "CPU" if cpu < total else "Tie"
    print(f"{N} | {cpu:.4f} | {total:.4f} | {kernel:.4f} | {winner}")

N | CPU ms | GPU Total ms | GPU Kernel ms | Winner


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


100 | 0.0892 | 1.4616 | 0.0925 | CPU
1000 | 0.2813 | 0.5788 | 0.0741 | CPU
10000 | 2.3640 | 0.5226 | 0.0544 | GPU
100000 | 25.3881 | 0.7538 | 0.0694 | GPU
1000000 | 264.7805 | 2.8362 | 0.1352 | GPU
5000000 | nan | 12.8655 | 0.2743 | Tie
10000000 | nan | 24.8551 | 0.4659 | Tie


In [7]:
# DEFECTIVE KERNEL: Notice the missing 'if idx < n' check!
@cuda.jit
def defective_kernel(d_arr):
 idx = cuda.grid(1)
 # FORBIDDEN: Writing without bounds check!
 d_arr[idx] = 999.0
# TEST HARNESS
N = 1000
h_arr = np.zeros(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)
# Launch 4 blocks of 256 = 1024 threads
threads_per_block = 256
blocks_per_grid = 4
print("Launching defective kernel with 1024 threads on 1000-element array...")
defective_kernel[blocks_per_grid, threads_per_block](d_arr)
cuda.synchronize()
print("Execution finished (or did it crash?). Checking host copy...")
h_result = d_arr.copy_to_host()
print(f"Elements processed: {len(h_result)}. Last element: {h_result[-1]}")



Launching defective kernel with 1024 threads on 1000-element array...
Execution finished (or did it crash?). Checking host copy...
Elements processed: 1000. Last element: 999.0


In [8]:
import numpy as np
import time
from numba import cuda

@cuda.jit
def naive_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        d_total[0] += d_arr[idx]

@cuda.jit
def atomic_sum_kernel(d_arr, d_total):
    idx = cuda.grid(1)
    if idx < d_arr.shape[0]:
        cuda.atomic.add(d_total, 0, d_arr[idx])

N = 50000
h_arr = np.ones(N, dtype=np.float32)
d_arr = cuda.to_device(h_arr)

threads = 256
blocks = (N + threads - 1) // threads

for trial in range(1, 6):
    d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
    naive_sum_kernel[blocks, threads](d_arr, d_total)
    cuda.synchronize()
    result = d_total.copy_to_host()[0]
    deviation = 50000.0 - result
    print(f"Trial {trial}: Sum={result}, Deviation={deviation}")

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
atomic_sum_kernel[blocks, threads](d_arr, d_total)
cuda.synchronize()
atomic_ms = (time.perf_counter() - t0) * 1000
atomic_result = d_total.copy_to_host()[0]

print("Atomic Sum Output:", atomic_result)
print("Atomic Kernel Time (ms):", atomic_ms)

d_total = cuda.to_device(np.zeros(1, dtype=np.float32))
t0 = time.perf_counter()
naive_sum_kernel[blocks, threads](d_arr, d_total)
cuda.synchronize()
naive_ms = (time.perf_counter() - t0) * 1000

print("Naive Kernel Time (ms):", naive_ms)

Trial 1: Sum=2.0, Deviation=49998.0
Trial 2: Sum=2.0, Deviation=49998.0
Trial 3: Sum=2.0, Deviation=49998.0
Trial 4: Sum=2.0, Deviation=49998.0
Trial 5: Sum=2.0, Deviation=49998.0
Atomic Sum Output: 50000.0
Atomic Kernel Time (ms): 160.43256699981612
Naive Kernel Time (ms): 0.5420869999852584


In [9]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    # Include GPU name
    h.update(cuda.get_current_device().name)

    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())

    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest


# Your Student ID
STUDENT_ID = "230103142"

# Replace with your actual crossover point from Section 2
CROSSOVER_N = 100000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

NameError: name 'd_canvas' is not defined

In [10]:
import numpy as np
import math
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024
canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + threads_2d[0] - 1) // threads_2d[0],
    (HEIGHT + threads_2d[1] - 1) // threads_2d[1]
)

radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2
)
cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print("Grid dimensions:", blocks_2d)
print("Total blocks:", blocks_2d[0] * blocks_2d[1])
print("Matrix shape:", result_canvas.shape)
print("Center value:", result_canvas[512, 512])
print("Corner value:", result_canvas[0, 0])

Grid dimensions: (64, 64)
Total blocks: 4096
Matrix shape: (1024, 1024)
Center value: 1.0
Corner value: 0.0


In [11]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    # Include GPU name
    h.update(cuda.get_current_device().name)

    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())

    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest


# Your Student ID
STUDENT_ID = "230103142"

# Replace with your actual crossover point from Section 2
CROSSOVER_N = 100000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

TypeError: Strings must be encoded before hashing

In [12]:
import numpy as np
import math
from numba import cuda

@cuda.jit
def radial_vignette_2d(d_canvas, width, height, center_x, center_y):
    x, y = cuda.grid(2)

    if x < width and y < height:
        dx = float(x - center_x)
        dy = float(y - center_y)
        dist = math.sqrt(dx * dx + dy * dy)
        max_dist = math.sqrt(float(center_x**2 + center_y**2))
        intensity = 1.0 - (dist / max_dist)

        if intensity < 0.0:
            intensity = 0.0

        d_canvas[y, x] = intensity

WIDTH, HEIGHT = 1024, 1024

canvas = np.zeros((HEIGHT, WIDTH), dtype=np.float32)
d_canvas = cuda.to_device(canvas)

threads_2d = (16, 16)
blocks_2d = (
    (WIDTH + 15) // 16,
    (HEIGHT + 15) // 16
)

radial_vignette_2d[blocks_2d, threads_2d](
    d_canvas, WIDTH, HEIGHT, WIDTH // 2, HEIGHT // 2
)

cuda.synchronize()

result_canvas = d_canvas.copy_to_host()

print("Matrix shape:", result_canvas.shape)
print("Center value:", result_canvas[512, 512])

Matrix shape: (1024, 1024)
Center value: 1.0


In [13]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()
    h.update(student_id_str.strip().encode('utf-8'))

    # Include GPU name
    h.update(cuda.get_current_device().name)

    # Include sample of compute output
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())

    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest


# Your Student ID
STUDENT_ID = "230103142"

# Replace with your actual crossover point from Section 2
CROSSOVER_N = 100000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

TypeError: Strings must be encoded before hashing

In [14]:
import hashlib
def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
 h = hashlib.sha256()
 h.update(student_id_str.strip().encode('utf-8'))
 # Include GPU name
 h.update(cuda.get_current_device().name)
 # Include sample of compute output
 h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
 h.update(str(exp_table_crossover).encode('utf-8'))
 digest = h.hexdigest()[:16].upper()
 print("=" * 60)
 print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
 print("=" * 60)
 return digest
# REPLACE WITH YOUR ACTUAL STUDENT ID AND EMPIRICAL CROSSOVER N* VALUE
STUDENT_ID = 230103142
CROSSOVER_N = 100000 # Example: your value from Section 2
generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)


AttributeError: 'int' object has no attribute 'strip'

In [15]:
import hashlib
from numba import cuda

def generate_lab_checksum(student_id_str, d_canvas, exp_table_crossover):
    h = hashlib.sha256()

    h.update(student_id_str.strip().encode('utf-8'))
    h.update(cuda.get_current_device().name.encode('utf-8'))
    h.update(d_canvas.copy_to_host()[:10, :10].tobytes())
    h.update(str(exp_table_crossover).encode('utf-8'))

    digest = h.hexdigest()[:16].upper()

    print("=" * 60)
    print(f"OFFICIAL VERIFICATION CHECKSUM TOKEN: {digest}")
    print("=" * 60)

    return digest

STUDENT_ID = "230103142"
CROSSOVER_N = 10000

generate_lab_checksum(STUDENT_ID, d_canvas, CROSSOVER_N)

OFFICIAL VERIFICATION CHECKSUM TOKEN: A29D65B39D927789


'A29D65B39D927789'

In [6]:
!nvidia-smi

Fri Oct  9 05:27:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             33W /   70W |     235MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----